# Diagnóstico Bronze — steam_games_review

Tabla analizada: `steam_game_intelligence.bronze.steam_games_review`. Se comprueban nulos, claves repetidas y consistencia de los snapshots. El campo `reviews` es string según el esquema proporcionado; no se presupone su estructura interna.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y cobertura

In [ ]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT app_id) AS distinct_app_ids,
  COUNT_IF(app_id IS NULL) AS null_app_id_rows,
  COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows,
  COUNT_IF(reviews IS NULL) AS null_reviews_rows,
  COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews_rows,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.steam_games_review;

## 2. Nulos por columna

In [ ]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(app_id IS NULL) AS app_id,
    COUNT_IF(name IS NULL) AS name,
    COUNT_IF(reviews IS NULL) AS reviews,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steam_games_review
) UNPIVOT (null_count FOR column_name IN (app_id, name, reviews, _ingested_at, _source))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Duplicados exactos por clave y día de ingesta

In [ ]:
%sql
SELECT app_id, CAST(_ingested_at AS DATE) AS ingestion_date,
       COUNT(*) AS row_count,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT reviews) AS distinct_reviews
FROM steam_game_intelligence.bronze.steam_games_review
GROUP BY app_id, CAST(_ingested_at AS DATE)
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

## 4. Historial de snapshots por app_id

In [ ]:
%sql
SELECT app_id,
       COUNT(*) AS snapshot_rows,
       COUNT(DISTINCT CAST(_ingested_at AS DATE)) AS ingestion_days,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT reviews) AS distinct_reviews,
       MIN(_ingested_at) AS first_seen,
       MAX(_ingested_at) AS last_seen
FROM steam_game_intelligence.bronze.steam_games_review
WHERE app_id IS NOT NULL
GROUP BY app_id
HAVING COUNT(*) > 1
ORDER BY snapshot_rows DESC
LIMIT 100;

## 5. Irregularidades básicas

In [ ]:
%sql
SELECT
  COUNT_IF(app_id <= 0) AS invalid_app_id,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp,
  COUNT_IF(name IS NOT NULL AND TRIM(name) = '') AS blank_name,
  COUNT_IF(reviews IS NOT NULL AND TRIM(reviews) = '') AS blank_reviews,
  COUNT_IF(_source IS NOT NULL AND TRIM(_source) = '') AS blank_source
FROM steam_game_intelligence.bronze.steam_games_review;

## 6. Registros para inspección

In [ ]:
%sql
SELECT *
FROM steam_game_intelligence.bronze.steam_games_review
WHERE app_id <= 0
   OR name IS NULL OR TRIM(name) = ''
   OR reviews IS NULL OR TRIM(reviews) = ''
   OR _ingested_at IS NULL
LIMIT 200;